# Test Pseudo-GT Scoring (Clean)

This notebook scores a test submission against the frozen pseudo/manual test annotations only. It does not write into training data directories. Treat the result as local pseudo-GT regression scoring, not hidden leaderboard truth.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path("/Users/pio/Documents/AIENGINEERCOURSE/detectionproject")
PYTHON = Path("/Users/pio/.DataAnalysis/bin/python")

# Change only this path when scoring a new submission.
SUBMISSION = PROJECT_ROOT / "working/submission.csv"

# Frozen pseudo/manual GT. Do not feed this file into training unless you intentionally accept leakage.
PSEUDO_GT = PROJECT_ROOT / "working/test_annotations/test_annotations.csv"

# Options: "all", "modified", "reviewed"
IMAGE_FILTER = "all"
IOU_THRESHOLD = 0.75

OUT_DIR = PROJECT_ROOT / "working/reports/test_pseudo_gt_scores" / f"notebook_{SUBMISSION.stem}_{IMAGE_FILTER}_{datetime.now():%Y%m%d_%H%M%S}"
OUT_DIR

In [ ]:
cmd = [
    str(PYTHON),
    str(PROJECT_ROOT / "tools/score_submission_against_pseudo_gt.py"),
    "--submission", str(SUBMISSION),
    "--pseudo-gt", str(PSEUDO_GT),
    "--image-filter", IMAGE_FILTER,
    "--iou-threshold", str(IOU_THRESHOLD),
    "--out-dir", str(OUT_DIR),
]
print("run:", " ".join(cmd))
result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

In [ ]:
summary = json.loads((OUT_DIR / "summary.json").read_text())
summary

In [ ]:
coco = pd.read_csv(OUT_DIR / "coco_map_summary.csv")
display(coco)

per_class = pd.read_csv(OUT_DIR / f"per_class_iou{int(IOU_THRESHOLD * 100):02d}.csv")
display(per_class.sort_values(["f1", "support"], ascending=[True, False]).head(20))

matches = pd.read_csv(OUT_DIR / f"matches_iou{int(IOU_THRESHOLD * 100):02d}.csv")
display(matches["match_type"].value_counts().rename_axis("match_type").reset_index(name="count"))

In [ ]:
png = OUT_DIR / f"confusion_matrix_iou{int(IOU_THRESHOLD * 100):02d}.png"
display(Image(filename=str(png)))
print(OUT_DIR)